# Log model artifacts to MLflow

Initialize the DagsHub MLflow tracking connection, discover every file in `model_development/artifacts`, and log the files to a single MLflow run.

In [ ]:
from pathlib import Path
import dagshub
import mlflow

dagshub.init(
    repo_owner='vijaytakbhate2002',
    repo_name='credit-card-fraud-transaction-model',
    mlflow=True,
)

In [ ]:
# Support running this notebook from either the project root or model_development/.
artifact_dir = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "model_development" / "artifacts").is_dir()),
    None,
)
if artifact_dir is None and (Path.cwd() / "artifacts").is_dir():
    artifact_dir = Path.cwd()
else:
    artifact_dir = artifact_dir / "model_development" / "artifacts" if artifact_dir else None

if artifact_dir is None:
    raise FileNotFoundError("Could not locate the model_development/artifacts folder.")

artifact_files = sorted(path for path in artifact_dir.rglob("*") if path.is_file())
if not artifact_files:
    raise FileNotFoundError(f"No model artifacts found in {artifact_dir}")

print(f"Found {len(artifact_files)} artifact file(s):")
for artifact_file in artifact_files:
    print(f"- {artifact_file.relative_to(artifact_dir)} ({artifact_file.stat().st_size:,} bytes)")

mlflow.set_experiment("fraud-detection-model-artifacts")
with mlflow.start_run(run_name="log-model-artifacts"):
    mlflow.log_param("artifact_count", len(artifact_files))
    mlflow.set_tag("artifact_source", str(artifact_dir))
    mlflow.log_artifacts(str(artifact_dir), artifact_path="model_artifacts")

print("All model artifact files were logged under the MLflow artifact path: model_artifacts")